# 📖 Notebook 3: Analytics & Click Tracking

A URL shortener isn't just about redirects — it's also about **knowing who clicks**.
Bitly's real value is analytics: click counts, geographic data, referrer tracking,
and time-series trends.

In this notebook we'll build a click tracking system and explore how to query
analytics data efficiently.

## Learning Objectives

By the end of this notebook you'll understand:
- How to record click events without slowing down redirects
- The difference between synchronous and asynchronous tracking
- How to use Redis for real-time counters (fast writes)
- How to query PostgreSQL for detailed analytics (rich queries)
- Why write-behind (Redis → DB) is perfect for analytics

## 🛠️ Setup

Make sure Docker is running:

```bash
cd system-designs/bitly
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `bitly_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import json
import random
from datetime import datetime, timedelta

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "bitly_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM clicks")
    click_count = cursor.fetchone()[0]
    conn.close()
    print(f"✅ Connected to PostgreSQL ({click_count} clicks in seed data)")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Analytics Challenge

Every time someone clicks a short URL, we want to record:
- **When** they clicked (timestamp)
- **Where** they came from (referrer — was it Twitter? Facebook? Direct?)
- **What device** they used (user agent)
- **What country** they're in (from IP geolocation)

But here's the tension: **recording this data must not slow down the redirect**.
The user expects to be redirected in < 100ms. Writing to a database takes time.

Two approaches:
1. **Synchronous** — write to DB before responding (simple but slow)
2. **Asynchronous** — respond first, write to DB later (fast but complex)

---
## Approach 1: Synchronous Click Tracking

The simplest approach: insert a click row into PostgreSQL before returning the redirect.

In [ ]:
def record_click_sync(short_code: str, referrer: str = None,
                       user_agent: str = None, ip: str = None,
                       country: str = None):
    """
    Record a click synchronously — write to PostgreSQL immediately.
    This happens BEFORE the redirect response is sent.
    """
    conn = get_db_connection()
    conn.autocommit = True
    cursor = conn.cursor()
    cursor.execute(
        """INSERT INTO clicks (short_code, referrer, user_agent, ip_address, country)
           VALUES (%s, %s, %s, %s, %s)""",
        (short_code, referrer, user_agent, ip, country)
    )
    conn.close()

def redirect_with_sync_tracking(short_code: str) -> dict:
    """
    Redirect + synchronous click recording.
    The user waits for BOTH the lookup AND the click insert.
    """
    # Look up the URL (using cache)
    redis_client = get_redis_client()
    cache_key = f"url:{short_code}"
    cached = redis_client.get(cache_key)
    
    if cached:
        long_url = cached
    else:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT long_url FROM urls WHERE short_code = %s", (short_code,))
        row = cursor.fetchone()
        conn.close()
        if row is None:
            return {"error": "not_found"}
        long_url = row[0]
        redis_client.setex(cache_key, 3600, long_url)
    
    # Record click BEFORE responding (synchronous)
    record_click_sync(
        short_code,
        referrer="https://twitter.com",
        user_agent="Mozilla/5.0",
        ip="192.168.1.100",
        country="US"
    )
    
    return {"url": long_url}

# Measure sync tracking latency
# Warm cache first
r = get_redis_client()
r.setex("url:abc123", 3600, "https://www.example.com")

sync_times = []
for _ in range(100):
    start = time.time()
    redirect_with_sync_tracking("abc123")
    sync_times.append((time.time() - start) * 1000)

avg_sync = sum(sync_times) / len(sync_times)
print(f"📊 Synchronous Tracking (100 requests):")
print(f"   Average latency: {avg_sync:.2f} ms")
print(f"   This includes: cache lookup + DB insert for the click")
print()
print("⚠️  The DB write adds latency to every redirect!")

---
## Approach 2: Asynchronous Tracking with Redis

Better approach: **respond immediately**, track the click in Redis, and flush to
PostgreSQL periodically in the background.

```
Click arrives → Look up URL (Redis) → Respond with 302
                                    → Push click to Redis list (fast)
                                    → Background job flushes to PostgreSQL
```

Redis is perfect for this because:
- `INCR` for real-time counters is atomic and sub-millisecond
- `LPUSH` to a list acts as a queue for batch processing
- The user doesn't wait for PostgreSQL at all

In [ ]:
r = get_redis_client()

def record_click_async(short_code: str, referrer: str = None,
                        user_agent: str = None, ip: str = None,
                        country: str = None):
    """
    Record a click asynchronously using Redis.
    
    Two things happen:
    1. Increment a real-time counter (for instant stats)
    2. Push detailed click data to a Redis list (for batch DB write)
    """
    # Real-time counter: instant click count
    r.incr(f"clicks:{short_code}")
    
    # Detailed event: pushed to a list for later DB flush
    click_event = json.dumps({
        "short_code": short_code,
        "referrer": referrer,
        "user_agent": user_agent,
        "ip_address": ip,
        "country": country,
        "clicked_at": datetime.now().isoformat()
    })
    r.lpush("click_queue", click_event)

def redirect_with_async_tracking(short_code: str) -> dict:
    """
    Redirect + asynchronous click recording.
    The user only waits for the URL lookup, not the click write.
    """
    redis_client = get_redis_client()
    cache_key = f"url:{short_code}"
    cached = redis_client.get(cache_key)
    
    if cached:
        long_url = cached
    else:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT long_url FROM urls WHERE short_code = %s", (short_code,))
        row = cursor.fetchone()
        conn.close()
        if row is None:
            return {"error": "not_found"}
        long_url = row[0]
        redis_client.setex(cache_key, 3600, long_url)
    
    # Record click AFTER getting the URL (async via Redis)
    record_click_async(
        short_code,
        referrer="https://twitter.com",
        user_agent="Mozilla/5.0",
        ip="192.168.1.100",
        country="US"
    )
    
    return {"url": long_url}

# Measure async tracking latency
async_times = []
for _ in range(100):
    start = time.time()
    redirect_with_async_tracking("abc123")
    async_times.append((time.time() - start) * 1000)

avg_async = sum(async_times) / len(async_times)

print(f"⚡ Performance Comparison (100 requests):")
print(f"=" * 50)
print(f"  Sync tracking:  avg {avg_sync:.2f} ms  (DB write on every request)")
print(f"  Async tracking: avg {avg_async:.2f} ms  (Redis push only)")
print(f"  Speedup:        {avg_sync / avg_async:.1f}×")
print()
print(f"📊 Click queue length: {r.llen('click_queue')} events waiting to be flushed")
print(f"📊 Real-time counter: {r.get('clicks:abc123')} clicks for abc123")

---
## Background Flush: Redis Queue → PostgreSQL

The click events sitting in Redis need to be written to PostgreSQL eventually,
so we can run rich SQL queries on them. A background job pops events from the
Redis list and batch-inserts them into the `clicks` table.

This is the **Write-Behind** pattern: writes go to the fast store (Redis) first,
and get flushed to the durable store (PostgreSQL) asynchronously.

In [ ]:
def flush_clicks_to_db(batch_size: int = 100) -> int:
    """
    Pop click events from the Redis queue and batch-insert them into PostgreSQL.
    
    This runs as a background job (e.g., every 5 seconds or when queue is large).
    Returns the number of events flushed.
    """
    redis_client = get_redis_client()
    events = []
    
    # Pop up to batch_size events from the queue
    for _ in range(batch_size):
        event_json = redis_client.rpop("click_queue")
        if event_json is None:
            break  # queue is empty
        events.append(json.loads(event_json))
    
    if not events:
        return 0
    
    # Batch insert into PostgreSQL
    conn = get_db_connection()
    conn.autocommit = True
    cursor = conn.cursor()
    
    # Build a single INSERT with multiple value rows (much faster than N inserts)
    values = []
    params = []
    for e in events:
        values.append("(%s, %s, %s, %s, %s, %s)")
        params.extend([
            e["short_code"], e["clicked_at"], e.get("referrer"),
            e.get("user_agent"), e.get("ip_address"), e.get("country")
        ])
    
    sql = f"""INSERT INTO clicks (short_code, clicked_at, referrer, user_agent, ip_address, country)
              VALUES {', '.join(values)}"""
    cursor.execute(sql, params)
    conn.close()
    
    return len(events)

# Flush the events we accumulated
queue_size = r.llen("click_queue")
print(f"📥 Queue before flush: {queue_size} events")

flushed = flush_clicks_to_db(batch_size=200)
print(f"📤 Flushed {flushed} events to PostgreSQL")

remaining = r.llen("click_queue")
print(f"📥 Queue after flush: {remaining} events")

# Verify in DB
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM clicks")
total = cursor.fetchone()[0]
conn.close()
print(f"\n📊 Total clicks in PostgreSQL: {total}")

---
## Querying Analytics from PostgreSQL

Now that click data is in PostgreSQL, we can run powerful SQL queries that
Redis can't do (JOINs, GROUP BY, date ranges, etc.).

In [ ]:
conn = get_db_connection()
cursor = conn.cursor()

# ── Query 1: Total clicks per URL ──────────────────────────
print("📊 Top URLs by Click Count")
print("=" * 60)
cursor.execute("""
    SELECT c.short_code, u.long_url, COUNT(*) as clicks
    FROM clicks c
    JOIN urls u ON c.short_code = u.short_code
    GROUP BY c.short_code, u.long_url
    ORDER BY clicks DESC
    LIMIT 5
""")
for row in cursor.fetchall():
    code, url, clicks = row
    # Truncate long URLs for display
    short_url = url[:50] + "..." if len(url) > 50 else url
    print(f"  {code}  {clicks:>5} clicks  {short_url}")

conn.close()

In [ ]:
conn = get_db_connection()
cursor = conn.cursor()

# ── Query 2: Clicks by country ─────────────────────────────
print("🌍 Clicks by Country")
print("=" * 40)
cursor.execute("""
    SELECT country, COUNT(*) as clicks
    FROM clicks
    WHERE country IS NOT NULL
    GROUP BY country
    ORDER BY clicks DESC
    LIMIT 10
""")
for country, clicks in cursor.fetchall():
    bar = "█" * (clicks // 20)
    print(f"  {country}  {clicks:>5}  {bar}")

conn.close()

In [ ]:
conn = get_db_connection()
cursor = conn.cursor()

# ── Query 3: Clicks by referrer ────────────────────────────
print("🔗 Clicks by Referrer")
print("=" * 50)
cursor.execute("""
    SELECT 
        COALESCE(referrer, 'Direct') as source,
        COUNT(*) as clicks
    FROM clicks
    GROUP BY referrer
    ORDER BY clicks DESC
    LIMIT 8
""")
for source, clicks in cursor.fetchall():
    bar = "█" * (clicks // 20)
    print(f"  {source:<35} {clicks:>5}  {bar}")

conn.close()

In [ ]:
conn = get_db_connection()
cursor = conn.cursor()

# ── Query 4: Clicks over time (daily) ──────────────────────
print("📈 Daily Click Trend (last 14 days)")
print("=" * 55)
cursor.execute("""
    SELECT 
        DATE(clicked_at) as day,
        COUNT(*) as clicks
    FROM clicks
    WHERE clicked_at >= NOW() - INTERVAL '14 days'
    GROUP BY day
    ORDER BY day
""")

rows = cursor.fetchall()
if rows:
    max_clicks = max(r[1] for r in rows)
    for day, clicks in rows:
        bar_len = int(clicks / max_clicks * 30) if max_clicks > 0 else 0
        bar = "█" * bar_len
        print(f"  {day}  {clicks:>4}  {bar}")
else:
    print("  No data in the last 14 days")

conn.close()

In [ ]:
conn = get_db_connection()
cursor = conn.cursor()

# ── Query 5: Hourly distribution (when do people click?) ───
print("🕐 Click Distribution by Hour of Day")
print("=" * 50)
cursor.execute("""
    SELECT 
        EXTRACT(HOUR FROM clicked_at)::int as hour,
        COUNT(*) as clicks
    FROM clicks
    GROUP BY hour
    ORDER BY hour
""")

rows = cursor.fetchall()
if rows:
    max_clicks = max(r[1] for r in rows)
    for hour, clicks in rows:
        bar_len = int(clicks / max_clicks * 25) if max_clicks > 0 else 0
        bar = "█" * bar_len
        print(f"  {hour:02d}:00  {clicks:>4}  {bar}")

conn.close()

---
## Real-Time Counters vs SQL Analytics

We now have **two sources** of click data:

| Source | Speed | Use Case |
|--------|-------|----------|
| **Redis counter** (`clicks:abc123`) | Sub-millisecond | Dashboard showing live click count |
| **PostgreSQL** (`clicks` table) | Milliseconds | Detailed breakdowns by country, referrer, time |

Redis gives you the **total right now**. PostgreSQL gives you the **breakdown**.

In [ ]:
# Compare: real-time counter vs SQL count

r = get_redis_client()

# Speed test: Redis counter lookup
redis_times = []
for _ in range(100):
    start = time.time()
    count = r.get("clicks:abc123")
    redis_times.append((time.time() - start) * 1000)

# Speed test: SQL COUNT query
sql_times = []
for _ in range(100):
    start = time.time()
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM clicks WHERE short_code = 'abc123'")
    cursor.fetchone()
    conn.close()
    sql_times.append((time.time() - start) * 1000)

avg_redis = sum(redis_times) / len(redis_times)
avg_sql = sum(sql_times) / len(sql_times)

print("⏱️  Click Count Lookup Speed (100 reads):")
print("=" * 50)
print(f"  Redis GET:      {avg_redis:.3f} ms  (real-time counter)")
print(f"  SQL COUNT(*):   {avg_sql:.3f} ms  (full table scan)")
print(f"  Speedup:        {avg_sql / avg_redis:.1f}×")
print()
print("💡 Use Redis for 'how many clicks right now?'")
print("   Use SQL for 'show me clicks by country in the last 7 days'")

---
## Putting It All Together: Full Analytics Dashboard

Let's build a function that combines Redis (real-time) and PostgreSQL (detailed)
to produce a complete analytics summary for any short URL.

In [ ]:
def get_url_analytics(short_code: str) -> dict:
    """
    Get complete analytics for a short URL.
    
    Combines:
    - Redis for real-time total click count (fast)
    - PostgreSQL for detailed breakdowns (rich)
    """
    redis_client = get_redis_client()
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # Real-time total from Redis
    total_clicks = int(redis_client.get(f"clicks:{short_code}") or 0)
    
    # If Redis counter is 0, fall back to SQL count
    if total_clicks == 0:
        cursor.execute("SELECT COUNT(*) FROM clicks WHERE short_code = %s", (short_code,))
        total_clicks = cursor.fetchone()[0]
    
    # Top countries
    cursor.execute("""
        SELECT country, COUNT(*) FROM clicks
        WHERE short_code = %s AND country IS NOT NULL
        GROUP BY country ORDER BY COUNT(*) DESC LIMIT 5
    """, (short_code,))
    top_countries = [{"country": r[0], "clicks": r[1]} for r in cursor.fetchall()]
    
    # Top referrers
    cursor.execute("""
        SELECT COALESCE(referrer, 'Direct'), COUNT(*) FROM clicks
        WHERE short_code = %s
        GROUP BY referrer ORDER BY COUNT(*) DESC LIMIT 5
    """, (short_code,))
    top_referrers = [{"source": r[0], "clicks": r[1]} for r in cursor.fetchall()]
    
    # Clicks in last 24 hours
    cursor.execute("""
        SELECT COUNT(*) FROM clicks
        WHERE short_code = %s AND clicked_at >= NOW() - INTERVAL '24 hours'
    """, (short_code,))
    last_24h = cursor.fetchone()[0]
    
    conn.close()
    
    return {
        "short_code": short_code,
        "total_clicks": total_clicks,
        "clicks_24h": last_24h,
        "top_countries": top_countries,
        "top_referrers": top_referrers
    }

# Get analytics for our most popular URL
analytics = get_url_analytics("abc123")

print(f"📊 Analytics for /{analytics['short_code']}")
print("=" * 50)
print(f"  Total clicks:    {analytics['total_clicks']}")
print(f"  Last 24 hours:   {analytics['clicks_24h']}")
print()
print("  Top Countries:")
for c in analytics['top_countries']:
    print(f"    {c['country']}  {c['clicks']} clicks")
print()
print("  Top Referrers:")
for r_item in analytics['top_referrers']:
    print(f"    {r_item['source']:<30}  {r_item['clicks']} clicks")

## 🧹 Cleanup

In [ ]:
# Clean up Redis keys we created
r = get_redis_client()
for key_pattern in ["clicks:*", "url:*", "click_queue"]:
    keys = r.keys(key_pattern)
    if keys:
        r.delete(*keys)

# Clean up our async-tracked clicks from DB
# (keep the seed data — only delete clicks from 192.168.1.100)
conn = get_db_connection()
conn.autocommit = True
cursor = conn.cursor()
cursor.execute("DELETE FROM clicks WHERE ip_address = '192.168.1.100'")
conn.close()

print("🧹 Cleaned up Redis keys and test click data")

## 📚 Summary

### Key Takeaways

1. **Sync tracking is simple but slow** — DB write on every click adds latency
2. **Async tracking (Redis queue) is fast** — push to Redis, flush to DB in background
3. **Redis counters** give instant totals; **PostgreSQL** gives detailed breakdowns
4. **Write-Behind pattern**: fast store (Redis) first, durable store (DB) later
5. **Batch inserts** are much faster than individual inserts

### Complete Architecture

```
┌──────────┐     ┌──────────────────┐     ┌───────────┐
│  Client   │────→│  FastAPI Server   │────→│   Redis   │
│ (Browser) │←────│                  │     │  • Cache   │
└──────────┘ 302  │  POST /shorten   │     │  • Counter │
                   │  GET  /{code}    │     │  • Queue   │
                   └──────────────────┘     └─────┬─────┘
                            │                     │
                            │                     │ flush
                            ▼                     ▼
                   ┌──────────────────────────────────┐
                   │         PostgreSQL               │
                   │  • urls table (short→long)       │
                   │  • clicks table (analytics)      │
                   └──────────────────────────────────┘
```

### What We Built in This Lab Series

| Notebook | Topic | Key Pattern |
|----------|-------|-------------|
| 1 | URL encoding & hash generation | Counter + Base62 |
| 2 | Redirect service with caching | Cache-Aside |
| 3 | Analytics & click tracking | Write-Behind (async) |

### Further Reading

- [Hello Interview — Design Bitly](https://www.hellointerview.com/learn/system-design/problem-breakdowns/bitly)
- [Redis INCR documentation](https://redis.io/docs/latest/commands/incr/)
- [PostgreSQL Indexing](https://www.postgresql.org/docs/current/indexes.html)
- [FastAPI documentation](https://fastapi.tiangolo.com/)